# Haya Skin Analyzer — YOLOv8 Training v2
**Run on Kaggle with T4 GPU** (Settings → Accelerator → GPU T4 x1)

Datasets: Acne · Dark Circles · Wrinkles
Target time: under 3 hours

In [ ]:
# Cell 1 — Install
!pip install ultralytics roboflow -q
import ultralytics
ultralytics.checks()

In [ ]:
# Cell 2 — Config
ROBOFLOW_API_KEY = "Dej9oDfQTdZXUPl6xeQe"
EPOCHS     = 60
MODEL_SIZE = "yolov8n"
IMG_SIZE   = 416
BATCH      = 32

In [ ]:
# Cell 3 — Download 3 focused datasets and merge them
from roboflow import Roboflow
import os, shutil, yaml
from pathlib import Path

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

datasets_to_download = [
    ("xzesoxee",                "acne-detection-9z3qf",  3),   # Acne / pimples
    ("skin-condition-detection", "dark-circles-19mqw",    1),   # Dark circles
    ("wrinkle-old-bucket",       "wrinkle-rkcym",        17),   # Wrinkles
]

downloaded = []
for workspace, project_name, version_num in datasets_to_download:
    try:
        print(f"Downloading {project_name}...")
        ds = rf.workspace(workspace).project(project_name).version(version_num).download("yolov8")
        downloaded.append(ds.location)
        print(f"  OK: {ds.location}")
    except Exception as e:
        print(f"  FAILED {project_name}: {e}")

if not downloaded:
    raise RuntimeError("No datasets downloaded — check API key and project slugs.")

print(f"\nDownloaded {len(downloaded)}/3 datasets")

# ── Merge all into one folder ────────────────────────────────────
MERGED = Path("/kaggle/working/merged")
for split in ("train", "valid", "test"):
    (MERGED / split / "images").mkdir(parents=True, exist_ok=True)
    (MERGED / split / "labels").mkdir(parents=True, exist_ok=True)

all_classes = []
for ds_path in downloaded:
    ds_path = Path(ds_path)
    yaml_file = next(ds_path.glob("*.yaml"), None)
    if yaml_file:
        with open(yaml_file) as f:
            dy = yaml.safe_load(f)
        all_classes.extend(dy.get("names", []))
    for split in ("train", "valid", "test"):
        for img in (ds_path / split / "images").glob("*"):
            shutil.copy(img, MERGED / split / "images" / img.name)
        for lbl in (ds_path / split / "labels").glob("*.txt"):
            shutil.copy(lbl, MERGED / split / "labels" / lbl.name)

# Deduplicate class names
unique_classes = list(dict.fromkeys(all_classes))

DATA_YAML = str(MERGED / "data.yaml")
with open(DATA_YAML, "w") as f:
    yaml.dump({
        "path":  str(MERGED),
        "train": "train/images",
        "val":   "valid/images",
        "test":  "test/images",
        "nc":    len(unique_classes),
        "names": unique_classes,
    }, f)

print("\nImage counts:")
for split in ("train", "valid", "test"):
    n = len(list((MERGED / split / "images").glob("*")))
    print(f"  {split}: {n} images")
print(f"\nClasses ({len(unique_classes)}): {unique_classes}")
print("\nDataset ready.")

In [ ]:
# Cell 4 — Train  (target: ~2 hours on Kaggle T4)
from ultralytics import YOLO

model = YOLO(f"{MODEL_SIZE}.pt")

results = model.train(
    data        = DATA_YAML,
    epochs      = EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = BATCH,
    patience    = 20,
    save        = True,
    save_period = 5,
    project     = "/kaggle/working",
    name        = "haya_skin_v2",
    exist_ok    = True,
    device      = 0,

    optimizer    = "AdamW",
    lr0          = 0.001,
    lrf          = 0.01,
    warmup_epochs= 3,
    weight_decay = 0.0005,

    augment  = True,
    mosaic   = 1.0,
    mixup    = 0.1,
    fliplr   = 0.5,
    degrees  = 10.0,
    hsv_s    = 0.5,
    hsv_v    = 0.3,

    val     = True,
    plots   = True,
    verbose = False,
)

print("Training complete.")

In [ ]:
# Cell 5 — Save, validate, and prepare for download
import shutil, glob, os
from ultralytics import YOLO

# Find best.pt
candidates = glob.glob("/kaggle/working/**/best.pt", recursive=True)
if not candidates:
    candidates = glob.glob("/kaggle/working/**/last.pt", recursive=True)
if not candidates:
    raise FileNotFoundError("No model found — training may have crashed. Check logs.")

shutil.copy(candidates[0], "/kaggle/working/best.pt")
size_mb = os.path.getsize("/kaggle/working/best.pt") / 1024 / 1024
print(f"best.pt saved ({size_mb:.1f} MB)")

# Validate
m = YOLO("/kaggle/working/best.pt")
metrics = m.val(data=DATA_YAML)
print(f"\nmAP50:     {metrics.box.map50:.3f}  (good > 0.50, great > 0.65)")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")
print(f"Classes:   {m.names}")

print("\n" + "="*50)
print("DONE. Click Save Version (top right) then")
print("go to Output tab and download best.pt")
print("Place it at: backend/app/models/skin_model.pt")
print("="*50)